Metrics (17)

Character level (letters and symbols of the text, spaces removed) - same as Kazmi:
- ttr — type/token ratio
- maas_ttr — Maas length-adjusted TTR
- msttr — mean segmental TTR (window 25)
- mattr — moving-average TTR
- mtld — measure of textual lexical diversity
- mtld_ma — moving-average MTLD
- vocd — vocd-D lexical diversity
- yulek — Yule’s K (repetition)

Word level (spaCy words, lowercased, punctuation removed) - same 8 measures with same settings:
- word_ttr, word_maas_ttr, word_msttr, word_mattr, word_mtld, word_mtld_ma, word_vocd, word_yulek

Other:
- lex_den — content/function word ratio

In [ ]:
import sys
import importlib
import spacy

importlib.invalidate_caches() # see packages installed after the kernel started
# download only if missing (uses the kernel's own python)
if not spacy.util.is_package("en_core_web_trf"):
    !{sys.executable} -m spacy download en_core_web_trf
print("en_core_web_trf installed:", spacy.util.is_package("en_core_web_trf"))

In [ ]:
import spacy
from spacy.tokens import Doc
from lexicalrichness import LexicalRichness

# en_core_web_trf - same parser as all other metric notebooks, lex_den uses its POS tags
# character and word tokens do not depend on the model (word tokens use tokenizer flags is_punct / is_space)
# only pos_ (tagger + attribute_ruler) is needed, so ner and lemmatizer are disabled
spacy.prefer_gpu() # uses the Mac GPU (MPS) if available
nlp = spacy.load("en_core_web_trf", disable=["ner", "lemmatizer"])
print(f"Language model: {nlp.meta['lang']}_{nlp.meta['name']}")
print(f"Version: {nlp.meta['version']}")
print(f"Pipe names: {nlp.pipe_names}")

CONTENT_POS = {"NOUN", "PROPN", "VERB", "ADJ", "ADV"}
FUNCTION_POS = {"DET", "ADP", "AUX", "PRON", "CCONJ", "SCONJ", "PART"}
CD_NAMES = ["ttr", "maas_ttr", "msttr", "mattr", "mtld", "mtld_ma", "vocd", "yulek"]
WORD_CD_NAMES = ["word_" + name for name in CD_NAMES]

def get_char_tokens(text):
    #character level - every character except space is one token (same as Kazmi)
    char_tokens = []
    for c in text:
        if c != " ":
            char_tokens.append(c)
    return char_tokens

def get_word_tokens(doc):
    #word level - spaCy words, lowercased, punctuation and whitespace removed
    #is_punct / is_space come from the tokenizer (not the tagger), so tokens do not depend on the tagger
    word_tokens = []
    for token in doc:
        if token.is_punct or token.is_space:
            continue
        word_tokens.append(token.text.lower())
    return word_tokens

def lexical_diversity(tokens):
    #8 diversity measures on a list of tokens - used for both character tokens and word tokens with same settings
    result = {}
    for name in CD_NAMES:
        result[name] = f"{0:.5f}"

    if len(tokens) >= 2:
        lex = LexicalRichness(tokens, preprocessor=None, tokenizer=None)

        try:
            result["ttr"] = f"{float(lex.ttr):.5f}"
        except (ValueError, ZeroDivisionError):
            pass

        try:
            result["maas_ttr"] = f"{float(lex.Maas):.5f}"
        except (ValueError, ZeroDivisionError):
            pass

        try:
            if len(tokens) > 25:
                result["msttr"] = f"{float(lex.msttr(segment_window=25)):.5f}"
            else:
                result["msttr"] = result["ttr"]
        except (ValueError, ZeroDivisionError):
            pass

        try:
            ww = min(25, len(tokens))
            if ww >= 1:
                result["mattr"] = f"{float(lex.mattr(window_size=ww)):.5f}"
        except (ValueError, ZeroDivisionError):
            pass

        mtld_val = 0.0
        try:
            mtld_val = float(lex.mtld(threshold=0.72))
            result["mtld"] = f"{mtld_val:.5f}"
        except (ValueError, ZeroDivisionError):
            pass

        try:
            if len(tokens) >= 50:
                vals = []
                step = 12
                start = 0
                while start <= len(tokens) - 25:
                    chunk = tokens[start:start + 25]
                    sub = LexicalRichness(chunk, preprocessor=None, tokenizer=None)
                    vals.append(float(sub.mtld(threshold=0.72)))
                    start += step
                if vals:
                    result["mtld_ma"] = f"{(sum(vals) / len(vals)):.5f}"
                else:
                    result["mtld_ma"] = f"{mtld_val:.5f}"
            else:
                result["mtld_ma"] = f"{mtld_val:.5f}"
        except (ValueError, ZeroDivisionError):
            result["mtld_ma"] = f"{mtld_val:.5f}"

        try:
            ntok = min(50, len(tokens) - 1)
            if ntok >= 35:
                result["vocd"] = f"{float(lex.vocd(ntokens=ntok, within_sample=100, iterations=3, seed=42)):.5f}"
        except (ValueError, ZeroDivisionError, RuntimeError):
            pass

        try:
            result["yulek"] = f"{float(lex.yulek):.5f}"
        except (ValueError, ZeroDivisionError):
            pass

    return result

def lexical_density(doc):
    #content words / function words, from POS tags
    n_content = 0
    n_function = 0
    for token in doc:
        if token.pos_ in CONTENT_POS:
            n_content += 1
        if token.pos_ in FUNCTION_POS:
            n_function += 1
    if n_function == 0:
        return f"{0:.5f}"
    return f"{(n_content / n_function):.5f}"

def calculate_kazmi_lexical(text):
    #text can be a raw string or an already parsed doc (from nlp.pipe), so the paragraph is never parsed twice
    doc = text if isinstance(text, Doc) else nlp(text or "")
    result = lexical_diversity(get_char_tokens(doc.text)) #character level, names without prefix (same as before)
    word_result = lexical_diversity(get_word_tokens(doc)) #word level, names with word_ prefix
    for name in CD_NAMES:
        result["word_" + name] = word_result[name]
    result["lex_den"] = lexical_density(doc)
    return result

In [ ]:
from pathlib import Path
import csv

LEVEL = "paragraph"  # "paragraph" or "sentence" - which data/<LEVEL>/raw-csv to read and data/<LEVEL>/features/ to write
raw_csv_path = Path(f"../../data/{LEVEL}/raw-csv")
# old values (character level only, en_core_web_sm) are in archive-pooled-sm-2026-11-11/kazmi-lexical-csv
out_csv_path = Path(f"../../data/{LEVEL}/features/kazmi-lexical-csv")
out_csv_path.mkdir(parents=True, exist_ok=True)

feature_names = CD_NAMES + WORD_CD_NAMES + ["lex_den"]

for csv_file in sorted(raw_csv_path.glob("*.csv")):
    out_csv_file = out_csv_path / csv_file.name
    if out_csv_file.exists() and out_csv_file.stat().st_size > 0: # Skip if already exist
       continue
    print(f"Working on file: {csv_file.name}")
    with csv_file.open("r",encoding="utf-8") as cf:
       csv_reader = csv.DictReader(cf)
       column_names = list(csv_reader.fieldnames)
       column_names = column_names[:-1] + feature_names + [column_names[-1]]
       rows = list(csv_reader)
    # write to a temp file and rename at the end, so a half written file is never skipped on restart
    tmp_csv_file = out_csv_file.with_suffix(".tmp")
    with tmp_csv_file.open("w",encoding="utf-8") as of:
       csv_writer = csv.DictWriter(of,fieldnames=column_names)
       csv_writer.writeheader()
       # nlp.pipe parses texts in batches - much faster than calling nlp(text) for every row
       for row, doc in zip(rows, nlp.pipe([row["text"] or "" for row in rows], batch_size=64)):
          feats = calculate_kazmi_lexical(doc)
          for name in feature_names:
             row[name] = feats[name]
          csv_writer.writerow(row)
    tmp_csv_file.replace(out_csv_file)
    # break # During demo we can uncomment and providea demo path

In [ ]:
# TEST KAZMI LEXICAL CODE

text = "Use a dehydrator to dry them if you have one. Set your dehydrator to 150 F degrees. Lay the almonds in a single layer on the dehydrator trays. Place the almonds in the dehydrator for 12 to 24 hours, or until crispy."
print(calculate_kazmi_lexical(text))
text = "Beyoncé and Rihanna are tied for the second - most #1 entries on the Billboard Hot 100 chart, with 5 each."
print(calculate_kazmi_lexical(text))

# character tokens vs word tokens of the same text
doc = nlp(text)
print("char tokens:", len(get_char_tokens(text)), get_char_tokens(text)[:12], "...")
print("word tokens:", len(get_word_tokens(doc)), get_word_tokens(doc))
feats = calculate_kazmi_lexical(text)
print(f"ttr: character level {feats['ttr']} vs word level {feats['word_ttr']}")

# word ttr by hand = distinct words / all words
words = get_word_tokens(doc)
assert abs(float(feats["word_ttr"]) - len(set(words)) / len(words)) < 1e-4

# edge cases - empty text gives 0, not a division by zero
assert calculate_kazmi_lexical("")["word_ttr"] == "0.00000"
assert calculate_kazmi_lexical("")["ttr"] == "0.00000"
print("all lexical checks passed")